# Tutorial: Plan 001 Catalog Tour

This notebook is for developers reviewing Archiver's first implementation increment.

**Prerequisites:** run it from the repository with `uv run jupyter lab` and know basic Python.

By the end, you will be able to create a catalog, scan a directory safely, inspect the current state, find duplicate content, and see why failed scans cannot replace a successful result.

## Outline

1. Create a temporary source tree.
2. Scan it into a SQLite catalog.
3. Query current observations and duplicate groups.
4. Observe history through a rename.
5. Simulate a failed scan safely.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch

from archiver import Catalog, ScanFailure

workspace_context = TemporaryDirectory(prefix="archiver-notebook-")
workspace = Path(workspace_context.name)
source = workspace / "source"
source.mkdir()
(source / "reports").mkdir()
(source / "reports" / "summary.txt").write_text("Quarterly summary\n", encoding="utf-8")
(source / "copy-a.txt").write_bytes(b"same content")
(source / "copy-b.txt").write_bytes(b"same content")

catalog_path = workspace / "catalog.sqlite"
catalog = Catalog.create(catalog_path)
print(f"Temporary workspace: {workspace}")
print("Source files:", sorted(path.relative_to(source).as_posix() for path in source.rglob("*") if path.is_file()))

In [ ]:
# show the files using the file system
list(source.iterdir())

## 1. Scan a directory

`scan_directory` reads regular-file bytes and writes observations only to the catalog database. It does not modify the source directory. The returned summary counts files, bytes, distinct content identities, and duplicate-content groups.


In [ ]:
summary = catalog.scan_directory(source)
summary

## 2. Inspect the current state

Paths are relative to the scanned location. Each observation also contains a SHA-256 `ContentId`; the shortened digest below makes the duplicate relationship easier to see.


In [ ]:
for observation in catalog.current_files(source):
    print(
        f"{observation.relative_path.as_posix():<24} "
        f"{observation.size_bytes:>3} bytes  "
        f"{observation.content_id.digest[:12]}..."
    )

In [ ]:
duplicate_groups = catalog.duplicate_groups(source)
for group in duplicate_groups:
    print([observation.relative_path.as_posix() for observation in group])

assert len(duplicate_groups) == 1

## 3. A rename changes the path, not content identity

A second successful scan becomes the location's current state. Historical observations remain in the database, while the query below returns only the latest successful view.


In [ ]:
before_rename = catalog.current_files(source)
copy_a = source / "copy-a.txt"
copy_a.rename(source / "renamed-copy.txt")

catalog.scan_directory(source)
after_rename = catalog.current_files(source)

print("Before:", [item.relative_path.as_posix() for item in before_rename])
print("After: ", [item.relative_path.as_posix() for item in after_rename])
assert before_rename[0].content_id == catalog.find_by_content(source, before_rename[0].content_id)[0].content_id

In [ ]:
#expected_content_id = before_rename
copy_a_content_id = next(
    obs.content_id
    for obs in before_rename
    if obs.relative_path.as_posix() == "copy-a.txt"
)
copy_a_content_id.digest
print('-------------------- Before the rename ---- ')
for observation in before_rename:
    if(copy_a_content_id.digest==observation.content_id.digest):
        print(
            f"{observation.relative_path.as_posix():<24} "
            f"{observation.size_bytes:>3} bytes  "
            f"{observation.content_id.digest[:12]}..."
        )
print('-------------------- After the rename ---- ')
for observation in catalog.current_files(source):
    if(copy_a_content_id.digest==observation.content_id.digest):
        print(
            f"{observation.relative_path.as_posix():<24} "
            f"{observation.size_bytes:>3} bytes  "
            f"{observation.content_id.digest[:12]}..."
        )    


## 4. Failed scans do not replace current state

This deliberately replaces the hashing function for one call. It is a demonstration technique only: normal applications should never monkeypatch the library. The important property is that `current_files` remains unchanged when scanning fails.


In [ ]:
current_before_failure = catalog.current_files(source)

with patch("archiver.catalog.hash_file_stably", side_effect=OSError("demonstration failure")):
    try:
        catalog.scan_directory(source)
    except ScanFailure as error:
        print(error)

assert catalog.current_files(source) == current_before_failure
print("Current state was preserved.")

## Exercise

Add another file with the same bytes as `copy-b.txt`, scan again, and confirm that the existing duplicate group grows to three paths.

**Common pitfall:** Do not place a catalog database inside a real source tree unless necessary. Archiver excludes its own SQLite database and sidecars, but keeping the database beside the scanned data is easier to reason about.

**Extension:** Change the bytes of `reports/summary.txt`, scan again, and compare its new `ContentId` with the old observation.


In [ ]:
catalog.close()
workspace_context.cleanup()
print("Temporary workspace removed and catalog closed.")